In [ ]:
# Classical simulation of the period-finding step in Shor's algorithm
# The idea here is to measure how the classical approach scales with N,
# and then extrapolate to see how absurd it gets for RSA-sized numbers.
# Spoiler: it's very absurd.

import math
import os
import numpy as np
import time
import random
import pandas as pd

def gcd(a, b):
    # Euclidean algorithm
    while b:
        a, b = b, a % b
    return a

def is_prime(n):
    # trial division to check if prime
    if n < 2 or (n > 2 and n % 2 == 0):
        return False
    if n == 2:
        return True
    # only need to check odd numbers up to sqrt(n)
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

def factorize(N):
    # trial division to factorise N into prime factors to get answer first
    factors = []
    d, temp = 2, N
    while d * d <= temp:
        while temp % d == 0:
            factors.append(d)
            temp //= d
        d += 1
    if temp > 1:
        factors.append(temp)
    return factors

def is_semiprime(n):
    # check if n is two distinct primes (Shor's algorithm only works for semiprimes)
    f = factorize(n)
    return len(f) == 2 and f[0] != f[1]

def classical_order(x, N):
    # sequential period finding
    r, value, ops = 1, x % N, 0
    start = time.perf_counter()
    while value != 1:
        value = (value * x) % N
        r, ops = r + 1, ops + 1
        if r > N:
            # giving up, period is too large (shouldn't happen for valid inputs)
            elapsed = time.perf_counter() - start
            return None, ops, elapsed
    elapsed = time.perf_counter() - start
    return r, ops, elapsed

def analyze_single_iteration(N, a):
    # test one iteration of Shor's algorithm
    iter_start = time.perf_counter()
    
    # first check: maybe we got lucky and gcd(a, N) already gives a factor
    g = gcd(a, N)
    if g != 1:
        # free factor from lucky base
        iter_time = time.perf_counter() - iter_start
        return {
            'success': True, 'failure_reason': None,
            'period': None, 'period_ops': 0, 'period_time': 0.0,
            'total_ops': 1, 'iter_time': iter_time,
            'gcd_shortcut': True
        }
    
    # no free factor, do period finding
    r, period_ops, period_time = classical_order(a, N)
    total_ops = period_ops + 1  # add ops everytime
    
    # period needs to be even for Shor's algorithm
    if r is None or r % 2 != 0:
        iter_time = time.perf_counter() - iter_start
        return {
            'success': False, 'failure_reason': 'odd_period',
            'period': r, 'period_ops': period_ops, 'period_time': period_time,
            'total_ops': total_ops, 'iter_time': iter_time,
            'gcd_shortcut': False
        }
    
    # check if a^(r/2) mod N gives us trivial result
    x_half = pow(a, r // 2, N)
    total_ops += 1
    if x_half == N - 1:
        # error period, cant extract factor
        iter_time = time.perf_counter() - iter_start
        return {
            'success': False, 'failure_reason': 'trivial_factor',
            'period': r, 'period_ops': period_ops, 'period_time': period_time,
            'total_ops': total_ops + 1, 'iter_time': iter_time,
            'gcd_shortcut': False
        }
    
    # pass validation tests, extract factors
    p, q = gcd(x_half - 1, N), gcd(x_half + 1, N)
    total_ops += 2
    success = (1 < p < N) or (1 < q < N)
    total_ops += 2
    
    iter_time = time.perf_counter() - iter_start
    return {
        'success': success,
        'failure_reason': None if success else 'no_factor',
        'period': r, 'period_ops': period_ops, 'period_time': period_time,
        'total_ops': total_ops, 'iter_time': iter_time,
        'gcd_shortcut': False
    }

def run_iterations(N, num_iterations=10):
    # run multiple iterations of Shor's algorithm, random bases each time
    bases = [a for a in range(2, min(N, 10000)) if gcd(a, N) == 1]
    if not bases:
        return None
    
    num_bases = len(bases)
    
    # warmup run to remove timing offset
    _ = analyze_single_iteration(N, random.choice(bases))
    
    # tracking lists for all our metrics
    period_ops_list = []
    period_time_list = []
    period_list = []
    total_ops_list = []
    iter_time_list = []
    
    # counters for analysis
    success_count = 0
    fail_odd_period = 0
    fail_trivial_factor = 0
    fail_no_factor = 0
    gcd_shortcut_count = 0
    first_success_iter = None
    
    for i in range(num_iterations):
        r = analyze_single_iteration(N, random.choice(bases))
        
        period_ops_list.append(r['period_ops'])
        period_time_list.append(r['period_time'] * 1000)  # convert to ms
        if r['period'] is not None:
            period_list.append(r['period'])
        
        total_ops_list.append(r['total_ops'])
        iter_time_list.append(r['iter_time'] * 1000)
        
        if r['gcd_shortcut']:
            gcd_shortcut_count += 1
        if r['success']:
            success_count += 1
            if first_success_iter is None:
                first_success_iter = i + 1
        else:
            reason = r['failure_reason']
            if reason == 'odd_period':
                fail_odd_period += 1
            elif reason == 'trivial_factor':
                fail_trivial_factor += 1
            elif reason == 'no_factor':
                fail_no_factor += 1
    
    period_ops_arr = np.array(period_ops_list)
    period_time_arr = np.array(period_time_list)
    total_ops_arr = np.array(total_ops_list)
    iter_time_arr = np.array(iter_time_list)
    
    return {
        # basic info about the semiprime
        'N': N, 'num_bits': N.bit_length(), 'num_digits': len(str(N)),
        'factors': factorize(N), 'num_bases': num_bases,
        'num_iterations': num_iterations,
        
        # how often did we actually succeed
        'success_count': success_count,
        'success_rate': 100 * success_count / num_iterations,
        'fail_odd_period': fail_odd_period,
        'fail_odd_period_pct': 100 * fail_odd_period / num_iterations,
        'fail_trivial_factor': fail_trivial_factor,
        'fail_trivial_factor_pct': 100 * fail_trivial_factor / num_iterations,
        'fail_no_factor': fail_no_factor,
        'fail_no_factor_pct': 100 * fail_no_factor / num_iterations,
        'gcd_shortcut_count': gcd_shortcut_count,
        'gcd_shortcut_pct': 100 * gcd_shortcut_count / num_iterations,
        'iters_to_first_success': first_success_iter,
        
        # period finding stats 
        'avg_period_ops': period_ops_arr.mean(),
        'std_period_ops': period_ops_arr.std(),
        'min_period_ops': int(period_ops_arr.min()),
        'max_period_ops': int(period_ops_arr.max()),
        'total_period_ops': int(period_ops_arr.sum()),
        'avg_period_time_ms': period_time_arr.mean(),
        'std_period_time_ms': period_time_arr.std(),
        'total_period_time_ms': round(float(period_time_arr.sum()), 4),
        
        # the actual period values we found
        'avg_period_r': np.mean(period_list) if period_list else 0,
        'std_period_r': np.std(period_list) if period_list else 0,
        'min_period_r': int(min(period_list)) if period_list else 0,
        'max_period_r': int(max(period_list)) if period_list else 0,
        
        # full iteration timing (includes all the overhead, not just period finding)
        'avg_total_ops': total_ops_arr.mean(),
        'avg_iter_time_ms': iter_time_arr.mean(),
        'std_iter_time_ms': iter_time_arr.std(),
        'total_execution_time_ms': round(float(iter_time_arr.sum()), 4),
    }

def generate_semiprimes(min_n, max_n, num_samples):
    """
    Generate semiprimes spread evenly across bit lengths.
    We want good coverage at every scale, not just a bunch of small ones.
    """
    min_bits = min_n.bit_length()
    max_bits = max_n.bit_length()
    
    # first pass: enumerate semiprimes at each bit length (if the range isn't too huge)
    all_by_bits = {}
    for bits in range(min_bits, max_bits + 1):
        low = max(min_n, 2**(bits-1))
        high = min(max_n, 2**bits - 1)
        if high - low < 200000:
            all_by_bits[bits] = sorted([n for n in range(low, high+1) if is_semiprime(n)])
        else:
            all_by_bits[bits] = None  # too many to enumerate, will sample instead
    
    # distribute samples across bit levels, redistributing leftovers from small levels
    result = []
    remaining_quota = num_samples
    remaining_levels = max_bits - min_bits + 1
    
    for bits in range(min_bits, max_bits + 1):
        target = remaining_quota // remaining_levels
        low = max(min_n, 2**(bits-1))
        high = min(max_n, 2**bits - 1)
        
        if all_by_bits[bits] is not None:
            available = all_by_bits[bits]
            if len(available) <= target:
                chosen = available
            else:
                # pick evenly spaced samples
                indices = np.linspace(0, len(available)-1, target, dtype=int)
                chosen = [available[i] for i in indices]
        else:
            # range is too big to enumerate, so search near evenly-spaced targets
            targets = np.linspace(low, high, target + 10)
            found, seen = [], set()
            for t in targets:
                start = int(t) | 1  # start from an odd number
                for i in range(5000):
                    for c in [start + 2*i, max(low, start - 2*i)]:
                        if low <= c <= high and is_semiprime(c) and c not in seen:
                            found.append(c)
                            seen.add(c)
                            break
                    else:
                        continue
                    break
                if len(found) >= target:
                    break
            chosen = sorted(found)[:target]
        
        result.extend(chosen)
        remaining_quota -= len(chosen)
        remaining_levels -= 1
    
    return sorted(result)

def extrapolate(results):
    # fit power law and extrapolate to RSA size
    N_arr = np.array([r['N'] for r in results])
    log_N = np.log10(N_arr)
    log_ops = np.log10(np.array([r['avg_period_ops'] for r in results]))
    slope, intercept = np.polyfit(log_N, log_ops, 1)
    a = 10**intercept
    
    # R-squared to check how good the fit is (least mean square)
    r2 = 1 - np.sum((log_ops - (slope*log_N + intercept))**2) / np.sum((log_ops - np.mean(log_ops))**2)
    
    # use the largest tested N as our reference for time-per-op conversion
    ref = results[-1]
    ref_time, ref_ops = ref['avg_period_time_ms']/1000, ref['avg_period_ops']
    sec_per_year = 365.25 * 24 * 3600
    
    rsa = {}
    for name, bits in [('RSA-512', 512), ('RSA-1024', 1024), ('RSA-2048', 2048), ('RSA-4096', 4096)]:
        log_N_rsa = bits * np.log10(2)
        log_ops_rsa = intercept + slope * log_N_rsa
        log_time = log_ops_rsa + np.log10(ref_time) - np.log10(ref_ops) if ref_ops else -9
        log_years = log_time - np.log10(sec_per_year)
        rsa[name] = {'ops': log_ops_rsa, 'years': log_years}
    
    return {'exponent': slope, 'coefficient': a, 'log_a': intercept, 'r_squared': r2, 'rsa': rsa,
            'fitted_log_ops': intercept + slope * log_N, 'log_N': log_N, 'log_ops': log_ops}

def run_analysis():
    # tests semiprimes from 15 up to 100 million (50 iterations each), report uses this
    MIN_N, MAX_N = 15, 100_000_000
    BIT_SPLIT = 14  # where we switch from small to large treatment
    SAMPLES_SMALL, ITERS_SMALL = 1000, 50   # lots of small semiprimes
    SAMPLES_LARGE, ITERS_LARGE = 1000, 50   # fewer large ones (they take longer)
    
    small_max = 2**BIT_SPLIT - 1  
    large_min = 2**BIT_SPLIT      
    
    print(f"Generating semiprimes: {SAMPLES_SMALL} small (4-{BIT_SPLIT} bits), {SAMPLES_LARGE} large ({BIT_SPLIT+1}-27 bits)")
    N_small = generate_semiprimes(MIN_N, small_max, SAMPLES_SMALL)
    N_large = generate_semiprimes(large_min, MAX_N, SAMPLES_LARGE)
    print(f"Generated {len(N_small)} small + {len(N_large)} large = {len(N_small)+len(N_large)} total")
    
    results = []
    total = len(N_small) + len(N_large)
    start = time.time()
    
    # process small semiprimes first 
    for i, N in enumerate(N_small, 1):
        if i % 50 == 0 or i == 1:
            pct = 100 * i / total
            eta = (time.time() - start) / i * (total - i) if i else 0
            print(f"[{i}/{total}] {pct:.1f}% | Small range | ETA: {eta:.0f}s")
        r = run_iterations(N, ITERS_SMALL)
        if r and r['success_rate'] > 0:
            results.append(r)
    
    # now the large ones (these take much longer per semiprime)
    for i, N in enumerate(N_large, 1):
        j = len(N_small) + i
        if i % 10 == 0 or i == 1:
            pct = 100 * j / total
            eta = (time.time() - start) / j * (total - j) if j else 0
            print(f"[{j}/{total}] {pct:.1f}% | Large range | ETA: {eta:.0f}s")
        r = run_iterations(N, ITERS_LARGE)
        if r and r['success_rate'] > 0:
            results.append(r)
    
    fit = extrapolate(results)
    elapsed = time.time() - start
    print(f"\nDone in {elapsed:.1f}s")
    print(f"Scaling: O(N^{fit['exponent']:.2f}), R^2 = {fit['r_squared']:.3f}")
    print(f"RSA-2048: ~10^{fit['rsa']['RSA-2048']['ops']:.0f} ops, ~10^{fit['rsa']['RSA-2048']['years']:.0f} years")
    
    config = {
        'BIT_SPLIT': BIT_SPLIT,
        'N_small_count': len(N_small),
        'N_large_count': len(N_large),
        'ITERS_SMALL': ITERS_SMALL,
        'ITERS_LARGE': ITERS_LARGE,
    }
    
    return results, fit, config

def export_to_excel(results, fit, config, excel_path=None):
    # export into excel sheet
    if excel_path is None:
        cwd = os.getcwd()
        if os.path.basename(cwd) == "Comparison analysis circuits":
            excel_path = os.path.join(cwd, "Classical_Shor_Analysis_Results.xlsx")
        else:
            excel_path = os.path.join(cwd, "Comparison analysis circuits", "Classical_Shor_Analysis_Results.xlsx")
    
    BIT_SPLIT = config['BIT_SPLIT']
    
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as w:
            # sheet 1: all the raw results
            pd.DataFrame([{
                'N': r['N'],
                'Bits': r['num_bits'],
                'Digits': r['num_digits'],
                'Factors': 'x'.join(map(str, r['factors'])),
                'Num_Bases': r['num_bases'],
                'Iterations': r['num_iterations'],
                
                'Success_Rate_%': r['success_rate'],
                'Fail_Odd_Period_%': r['fail_odd_period_pct'],
                'Fail_Trivial_Factor_%': r['fail_trivial_factor_pct'],
                'Fail_No_Factor_%': r['fail_no_factor_pct'],
                'GCD_Shortcut_%': r['gcd_shortcut_pct'],
                'Iters_To_First_Success': r['iters_to_first_success'],
                
                'Avg_Period_r': round(r['avg_period_r'], 2),
                'Std_Period_r': round(r['std_period_r'], 2),
                'Min_Period_r': r['min_period_r'],
                'Max_Period_r': r['max_period_r'],
                
                'Avg_Period_Ops': round(r['avg_period_ops'], 2),
                'Std_Period_Ops': round(r['std_period_ops'], 2),
                'Min_Period_Ops': r['min_period_ops'],
                'Max_Period_Ops': r['max_period_ops'],
                'Total_Period_Ops': r['total_period_ops'],
                'Avg_Period_Time_ms': round(r['avg_period_time_ms'], 6),
                'Std_Period_Time_ms': round(r['std_period_time_ms'], 6),
                'Total_Period_Time_ms': r['total_period_time_ms'],
                
                'Avg_Total_Ops_Per_Iter': round(r['avg_total_ops'], 2),
                'Avg_Iter_Time_ms': round(r['avg_iter_time_ms'], 6),
                'Std_Iter_Time_ms': round(r['std_iter_time_ms'], 6),
                'Total_Execution_Time_ms': r['total_execution_time_ms'],
                
                # log-scale versions for the scaling plots
                'Log10_N': round(fit['log_N'][i], 4),
                'Log10_Avg_Period_Ops': round(fit['log_ops'][i], 4),
                'Log10_Avg_Period_Ops_Fitted': round(fit['fitted_log_ops'][i], 4),
                'Log10_Avg_Period_Time_s': round(np.log10(r['avg_period_time_ms'] / 1000), 4) if r['avg_period_time_ms'] > 0 else None
            } for i, r in enumerate(results)]).to_excel(w, sheet_name='Results', index=False)
            
            # sheet 2: the RSA extrapolation numbers
            pd.DataFrame([{
                'Key': k,
                'Bits': bits,
                'Log10_N': round(bits * np.log10(2), 1),
                'Log10_Ops': round(v['ops'], 1),
                'Log10_Years': round(v['years'], 1)
            } for (k, v), bits in zip(fit['rsa'].items(), [512, 1024, 2048, 4096])]).to_excel(w, sheet_name='RSA_Extrapolation', index=False)
            
            # sheet 3: fit parameters for reproducibility
            pd.DataFrame([
                {'Parameter': 'Exponent (b)', 'Value': round(fit['exponent'], 4), 'Description': 'Slope of log-log fit; ops scales as N^b'},
                {'Parameter': 'Coefficient (a)', 'Value': round(fit['coefficient'], 6), 'Description': 'Multiplier in ops = a * N^b'},
                {'Parameter': 'Log10(a)', 'Value': round(fit['log_a'], 4), 'Description': 'Y-intercept of log-log fit'},
                {'Parameter': 'R_squared', 'Value': round(fit['r_squared'], 4), 'Description': 'Goodness of fit (1.0 = perfect)'},
                {'Parameter': 'N_min', 'Value': results[0]['N'], 'Description': 'Smallest semiprime tested'},
                {'Parameter': 'N_max', 'Value': results[-1]['N'], 'Description': 'Largest semiprime tested'},
                {'Parameter': 'Num_Samples_Total', 'Value': len(results), 'Description': 'Total semiprimes tested'},
                {'Parameter': 'Num_Samples_Small', 'Value': config['N_small_count'], 'Description': f'Semiprimes in 4-{BIT_SPLIT} bit range'},
                {'Parameter': 'Num_Samples_Large', 'Value': config['N_large_count'], 'Description': f'Semiprimes in {BIT_SPLIT+1}-27 bit range'},
                {'Parameter': 'Iterations_Small', 'Value': config['ITERS_SMALL'], 'Description': f'Iterations per semiprime (4-{BIT_SPLIT} bits)'},
                {'Parameter': 'Iterations_Large', 'Value': config['ITERS_LARGE'], 'Description': f'Iterations per semiprime ({BIT_SPLIT+1}-27 bits)'},
            ]).to_excel(w, sheet_name='Fit_Parameters', index=False)
            
        print(f"Exported: {excel_path}")
    except Exception as e:
        print(f"Excel export failed: {e}")

def main():
    results, fit, config = run_analysis()
    export_to_excel(results, fit, config)

if __name__ == "__main__":
    main()